# Step 2. Data Understanding

In Step 1 I framed the problem. In Step 2 I check that the data can carry it, and I build the reference every later step reads from.

I answer three questions.

1. Where does the data come from, and can I trust it?
2. What is in the table, and is it clean?
3. What kind of thing is each column, and which columns matter for fairness?

The work here feeds Step 3, where I clean and prepare the data, and Step 5, where I audit it for bias.

In [ ]:
import sys
from pathlib import Path

# I add the repo root to the path so src/ is importable from notebooks/.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from src.data import (
    load_raw, load_binary,
    CONTINUOUS, BINARY_FLAGS, NOMINAL_CODED, COUNT_ORDINAL, LEAKAGE,
    SENSITIVE, TARGET,
)

df = load_raw()      # all 37 columns, names cleaned
bdf = load_binary()  # Dropout vs Graduate, with a 0/1 dropout column
print("raw", df.shape, "| binary", bdf.shape)

## 1. Where the Data Comes From

The data is the UCI set Predict Students' Dropout and Academic Success, from Realinho and colleagues at the Polytechnic Institute of Portalegre. It carries a CC BY 4.0 license, free to reuse with credit, under DOI 10.24432/C5MC89. I keep the full citation, funding, and license in data/README.md.

The file gathers several separate institutional databases into one table, one row per student. The providers state they cleaned it for anomalies, outliers, and missing values before release. That matters for the next section. The clean state I see here is their work, not a property of raw institutional records.

## 2. Dataset Overview

I take a first pass over the whole table. Size, column types, missing values, duplicate rows, the outcome split, and the spread of the true numbers.

In [ ]:
print("shape", df.shape)
print()
print("dtypes")
print(df.dtypes.value_counts())
print()
print("missing values total", int(df.isna().sum().sum()))
print("duplicate rows", int(df.duplicated().sum()))
print()
print("target split")
print(df[TARGET].value_counts())

In [ ]:
# I check the ranges and spread of the six true numbers. Order and distance are real here.
df[CONTINUOUS].describe().round(1)

I find no missing cells and no duplicate rows. This is the providers' cleaning noted above, not luck, so in Step 3 I validate ranges and keep real extremes like mature student ages, rather than re-cleaning. The outcome splits three ways, Graduate 2209, Dropout 1421, Enrolled 794. My binary frame drops Enrolled and works with 3630 students. The ranges look sound, with grades between 0 and 200 and ages that reach into mature entry.

## 3. Feature Types

The 37 columns are not one kind of thing. They fall into a few types, and each type needs different handling later. I name the types now to prevent a common mistake in Step 3.

Four types describe the model inputs.

True numbers. Six columns where the value is a real quantity and order and distance mean something, like admission grade and age.

Yes or no flags. Eight columns stored as 1 or 0, like gender, scholarship holder, and debtor.

Coded categories. Nine columns where an integer stands for a category, like course and mother's qualification. The number is a label, not an amount. A course coded 9500 is not greater than one coded 33, it is a different course. Treating these as numbers, scaling them or measuring plain correlation on them, is wrong, so in Step 3 I handle them as categories.

Ordered count. One column, application order, a rank from 0 first choice to 9 last choice.

Beyond these sit twelve curricular columns from the first and second semesters, which I hold out as leakage per Step 1. The Target column is the outcome.

In [ ]:
# I confirm the grouping covers all 37 columns.
groups = {
    "true number": CONTINUOUS,
    "yes or no flag": BINARY_FLAGS,
    "coded category": NOMINAL_CODED,
    "ordered count": COUNT_ORDINAL,
    "curricular (leakage)": LEAKAGE,
    "target": [TARGET],
}
for name, cols in groups.items():
    print(f"{name:22} {len(cols)}")
print("total", sum(len(c) for c in groups.values()))

## 4. The Data Dictionary

The dictionary lists every column with its type, its range or values, its unique count, and whether it is a sensitive attribute. I build it from the file in the next cell, so it stays true to the data and cannot drift from it. A grouped, plain language version follows for reading.

In [ ]:
# I build the dictionary from the data, so it matches the file exactly.
group_lookup = {c: name for name, cols in groups.items() for c in cols}

def describe_range(col):
    s = df[col]
    if col in CONTINUOUS or col in LEAKAGE:
        return f"{s.min():g} to {s.max():g}"
    if col in BINARY_FLAGS:
        return "0 or 1"
    if col in NOMINAL_CODED:
        return f"{s.nunique()} codes"
    if col in COUNT_ORDINAL:
        return f"{int(s.min())} to {int(s.max())}"
    if col == TARGET:
        return ", ".join(map(str, s.unique()))
    return ""

data_dict = pd.DataFrame([
    {
        "column": c,
        "group": group_lookup.get(c, "other"),
        "dtype": str(df[c].dtype),
        "unique": int(df[c].nunique()),
        "range or values": describe_range(c),
        "missing": int(df[c].isna().sum()),
        "sensitive": "yes" if c in SENSITIVE else "",
    }
    for c in df.columns
])
data_dict

Here are the same columns grouped, with plain meaning. The table above holds the exact types and ranges.

| group | columns | meaning |
|---|---|---|
| true number | Admission grade, Previous qualification (grade), Age at enrollment, Unemployment rate, Inflation rate, GDP | real quantities where order and distance matter |
| yes or no flag | Gender, Scholarship holder, Debtor, Tuition fees up to date, Displaced, Educational special needs, International, Daytime/evening attendance | stored as 1 or 0 |
| coded category | Course, Marital status, Application mode, Previous qualification, Nacionality, Mother's and Father's qualification, Mother's and Father's occupation | integer codes standing for categories, labels not amounts |
| ordered count | Application order | rank from 0 first choice to 9 last choice |
| curricular, leakage | twelve first and second semester records | performance after enrollment, held out of the model |
| target | Target | Dropout, Graduate, Enrolled, with Enrolled dropped for the binary frame |

The sensitive attributes for my Step 5 audit are Gender, Age at enrollment, Scholarship holder, and Debtor. I watch Tuition fees up to date too, for the reason the next section shows.

## 5. Sensitive Attributes, and a First Look at Disparity

Four attributes drive my fairness work in Step 5. Gender, age at enrollment, scholarship holder, and debtor. The next cell shows the dropout rate inside each group, against an overall rate near 39 percent. This is the baseline my Step 5 audit builds on.

In [ ]:
bdf["age band"] = pd.cut(
    bdf["Age at enrollment"], bins=[16, 20, 23, 30, 100],
    labels=["17 to 20", "21 to 23", "24 to 30", "31 plus"],
)

overall = bdf["dropout"].mean()
print("overall dropout rate", round(overall * 100, 1), "percent")
print()

def show_rate(col, labels=None):
    g = bdf.groupby(col, observed=True)["dropout"].agg(["mean", "size"])
    print(col)
    for idx, row in g.iterrows():
        name = labels.get(idx, idx) if labels else idx
        print(f"  {str(name):16} {row['mean'] * 100:5.1f} percent   n={int(row['size'])}")
    print()

show_rate("Gender", {1: "male", 0: "female"})
show_rate("Scholarship holder", {1: "scholarship", 0: "no scholarship"})
show_rate("Debtor", {1: "debtor", 0: "not debtor"})
show_rate("Tuition fees up to date", {1: "up to date", 0: "not up to date"})
show_rate("age band")

### Reading the Gaps

The gaps are real and large. Men drop out far more than women, 56 against 30 percent. Students without a scholarship drop out far more than holders, 48 against 14. Students with debt drop out far more than those without, 76 against 35. Older entrants drop out more than the youngest, near 66 percent in the 24 to 30 band against 26 in the 17 to 20 band. A model trained here can carry these gaps into its flags, so in Step 5 I measure them and work to reduce them.

One column stands apart, tuition fees up to date. Students not up to date drop out almost always, 94 percent. This is not a normal background fact. A student who has stopped paying is often a student already leaving, so this flag sits close to the outcome, like the semester records I held out in Step 1. I keep it for now but watch it. In Step 3 I test the model with and without it, so I measure its heavy influence rather than hide it.

## What Step 2 Settles

Five things carry into Step 3.

1. The dataset is cited and licensed, with the full record in data/README.md.
2. There are no missing values and no duplicate rows, a clean state the providers curated.
3. The 37 columns fall into four input types, plus twelve curricular columns I hold out as leakage.
4. The coded categories are labels, not amounts, so they need category handling, not scaling.
5. All four sensitive attributes show real disparity, and tuition status behaves like a near-outcome signal I watch.

In Step 3 I clean, explore, and engineer features on this base, routing each feature type the correct way.